# Notebook 1: RSNN Architecture Lab
## Systematic Architectural Exploration Across Three Datasets

**Purpose**: Train 8 architectural variants of the RSNN on SHD, PTB-XL (ECG), FI-2010 (LOB).
Save all 24+ checkpoints. Identify best architecture per dataset.

| ID | Modification | Reference |
|---|---|---|
| B0 | Base RSNN | Cramer et al. 2020 |
| B1 | ALIF (adaptive threshold) | Bellec et al. 2020, Nature Comms |
| B2 | Heterogeneous time constants | Perez-Nieves et al. 2021, Nature Comms |
| B3 | 2-layer RSNN | Cramer et al. 2020, Fig 8b |
| B4 | Input batch normalisation | Kim & Panda 2021, Frontiers Neurosci |
| B5 | Attention-weighted readout | Yao et al. 2023, NeurIPS |
| B6 | Scale (512 neurons) | Cramer et al. 2020 best effort |
| B7 | Best combo (data-driven) | Top-2 from B1-B6 per dataset |

**Output**: 24+ `.pth` checkpoints + `architecture_results.json`

**Hardware**: Kaggle P100/T4. Estimated ~5 hours total.

## 0. Setup

In [1]:
import sys, subprocess
print('Installing dependencies...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'snntorch', 'wfdb', '-q'])
print('Done.')

Installing dependencies...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 8.4 MB/s eta 0:00:00


Done.


In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import OrderedDict
import copy, time, json, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SAVE_DIR = '/kaggle/working/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
ALL_RESULTS = {}

Device: cuda
GPU: Tesla T4


## 1. Spike Encoding Methods

Four encoding methods for continuous signals. Adaptive delta calibrates
per-channel thresholds to the data distribution. Critical for FI-2010
where inter-step changes are O(1e-4).

**Adaptive delta ref**: Amir et al. 2017, CVPR.

In [3]:
class SpikeEncoder:
    @staticmethod
    def direct(signal):
        return signal

    @staticmethod
    def rate_coding(signal):
        sig_min = signal.min(axis=0, keepdims=True)
        sig_max = signal.max(axis=0, keepdims=True)
        probs = (signal - sig_min) / (sig_max - sig_min + 1e-8)
        return (np.random.rand(*signal.shape) < probs).astype(np.float32)

    @staticmethod
    def delta_modulation(signal, threshold=0.1):
        T, C = signal.shape
        spikes = np.zeros_like(signal)
        reference = signal[0].copy()
        for t in range(1, T):
            diff = signal[t] - reference
            up = diff > threshold
            down = diff < -threshold
            spikes[t, up] = 1.0
            spikes[t, down] = -1.0
            reference[up] = signal[t, up]
            reference[down] = signal[t, down]
        return spikes

    @staticmethod
    def temporal_contrast(signal, threshold=0.05):
        T, C = signal.shape
        spikes = np.zeros((T, C * 2), dtype=np.float32)
        for t in range(1, T):
            diff = signal[t] - signal[t - 1]
            spikes[t, :C] = (diff > threshold).astype(np.float32)
            spikes[t, C:] = (diff < -threshold).astype(np.float32)
        return spikes

    @staticmethod
    def adaptive_delta(signal, percentile=95):
        T, C = signal.shape
        diffs = np.abs(np.diff(signal, axis=0))
        thresholds = np.percentile(diffs, percentile, axis=0)
        thresholds = np.maximum(thresholds, 1e-8)
        spikes = np.zeros_like(signal)
        reference = signal[0].copy()
        for t in range(1, T):
            diff = signal[t] - reference
            up = diff > thresholds
            down = diff < -thresholds
            spikes[t, up] = 1.0
            spikes[t, down] = -1.0
            reference[up] = signal[t, up]
            reference[down] = signal[t, down]
        return spikes, thresholds

## 2. SNN Components

### 2.1 Surrogate Gradient

Fast sigmoid surrogate (Zenke & Ganguli 2018; Cramer et al. 2020 Eq. 8), slope beta=40.

In [4]:
class SurrogateSpike(torch.autograd.Function):
    beta = 40.0
    @staticmethod
    def forward(ctx, mem, threshold=1.0):
        ctx.save_for_backward(mem)
        ctx.threshold = threshold
        return (mem >= threshold).float()
    @staticmethod
    def backward(ctx, grad_output):
        mem, = ctx.saved_tensors
        v = mem - ctx.threshold
        grad = 1.0 / (1.0 + SurrogateSpike.beta * torch.abs(v)) ** 2
        return grad_output * grad, None

def spike_fn(x, threshold=1.0):
    return SurrogateSpike.apply(x, threshold)

### 2.2 LIF Layer (Base)

Standard LIF with current-based synapses (Cramer et al. 2020 Eqs. 5-6).
Supports both shared and per-neuron (heterogeneous) time constants.

In [5]:
class LIFLayer(nn.Module):
    # LIF neuron layer. Cramer et al. 2020 Eqs 5-6.
    # Supports heterogeneous tau (Perez-Nieves et al. 2021) via flag.
    def __init__(self, input_size, hidden_size, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 learnable_tau=False, dropout=0.0,
                 heterogeneous_tau=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout

        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)

        if heterogeneous_tau:
            log_tau_mem = torch.empty(hidden_size).uniform_(np.log(5.0), np.log(200.0))
            log_tau_syn = torch.empty(hidden_size).uniform_(np.log(2.0), np.log(100.0))
        else:
            log_tau_mem = torch.full((hidden_size,), np.log(tau_mem_init))
            log_tau_syn = torch.full((hidden_size,), np.log(tau_syn_init))

        if learnable_tau:
            self.log_tau_mem = nn.Parameter(log_tau_mem)
            self.log_tau_syn = nn.Parameter(log_tau_syn)
        else:
            self.register_buffer('log_tau_mem', log_tau_mem)
            self.register_buffer('log_tau_syn', log_tau_syn)

        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_mem))

    def forward(self, x):
        B, T, _ = x.shape
        alpha, beta = self.alpha, self.beta
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        spike_rec, mem_rec = [], []
        for t in range(T):
            syn = alpha * syn + self.W_ff(x[:, t])
            if self.recurrent:
                rec_spk = F.dropout(prev_spk, p=self.dropout,
                                    training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_spk)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem, threshold=1.0)
            spike_rec.append(spk)
            mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spike_rec, dim=1), torch.stack(mem_rec, dim=1)

### 2.3 ALIF Layer (B1)

Adaptive LIF. After each spike, threshold increases then decays exponentially.

**Reference**: Bellec et al. (2020), 'A solution to the learning dilemma for
recurrent networks of spiking neurons', Nature Communications 11, 3625.

**Implementation note**: Surrogate gradient at base threshold (1.0), adaptive
threshold applied as multiplicative mask. Standard approximation (snnTorch, Norse).
Full e-prop backward pass not implemented.

In [6]:
class ALIFLayer(nn.Module):
    # Adaptive LIF. Bellec et al. 2020.
    # a(t+1) = rho * a(t) + spk(t)
    # threshold(t) = 1.0 + beta_adapt * a(t)
    def __init__(self, input_size, hidden_size, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 tau_adapt=100.0, beta_adapt=0.1,
                 learnable_tau=False, dropout=0.0,
                 heterogeneous_tau=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout
        self.beta_adapt = beta_adapt
        self.rho = np.exp(-dt / tau_adapt)

        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)

        if heterogeneous_tau:
            log_tau_mem = torch.empty(hidden_size).uniform_(np.log(5.0), np.log(200.0))
            log_tau_syn = torch.empty(hidden_size).uniform_(np.log(2.0), np.log(100.0))
        else:
            log_tau_mem = torch.full((hidden_size,), np.log(tau_mem_init))
            log_tau_syn = torch.full((hidden_size,), np.log(tau_syn_init))

        if learnable_tau:
            self.log_tau_mem = nn.Parameter(log_tau_mem)
            self.log_tau_syn = nn.Parameter(log_tau_syn)
        else:
            self.register_buffer('log_tau_mem', log_tau_mem)
            self.register_buffer('log_tau_syn', log_tau_syn)

        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_mem))

    def forward(self, x):
        B, T, _ = x.shape
        alpha, beta = self.alpha, self.beta
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        adapt = torch.zeros(B, self.hidden_size, device=x.device)
        spike_rec, mem_rec = [], []
        for t in range(T):
            syn = alpha * syn + self.W_ff(x[:, t])
            if self.recurrent:
                rec_spk = F.dropout(prev_spk, p=self.dropout,
                                    training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_spk)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            threshold = 1.0 + self.beta_adapt * adapt
            spk_base = spike_fn(mem, threshold=1.0)
            spk = spk_base * (mem >= threshold).float()
            adapt = self.rho * adapt + spk
            spike_rec.append(spk)
            mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spike_rec, dim=1), torch.stack(mem_rec, dim=1)

### 2.4 Readout Layers

1. **Membrane readout + max-over-time** (base)
2. **Attention-weighted readout** (B5): learned per-timestep weights

**Attention ref**: Yao et al. 2023, NeurIPS (Spike-driven Transformer);
Yin et al. 2021, Nature Machine Intelligence.

In [7]:
class ReadoutLayer(nn.Module):
    # Leaky membrane readout (no spikes). Cramer et al. 2020.
    def __init__(self, input_size, output_size, tau_mem=20.0, dt=10.0):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size, bias=False)
        self.beta = np.exp(-dt / tau_mem)
        nn.init.kaiming_uniform_(self.fc.weight, nonlinearity='linear')

    def forward(self, x):
        B, T, _ = x.shape
        mem = torch.zeros(B, self.fc.out_features, device=x.device)
        mem_rec = []
        for t in range(T):
            mem = self.beta * mem + (1.0 - self.beta) * self.fc(x[:, t])
            mem_rec.append(mem)
        return torch.stack(mem_rec, dim=1)


class AttentionReadout(nn.Module):
    # Attention-weighted readout. Yao et al. 2023.
    def __init__(self, input_size, output_size):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size, bias=False)
        self.attn = nn.Linear(input_size, 1, bias=False)
        nn.init.kaiming_uniform_(self.fc.weight, nonlinearity='linear')

    def forward(self, x):
        logits = self.fc(x)  # [B, T, output]
        attn_weights = F.softmax(self.attn(x), dim=1)  # [B, T, 1]
        return logits, attn_weights

### 2.5 Complete SNN

Modular SNN covering all B0-B7 variants through constructor kwargs.
Single class, no code duplication.

In [8]:
class SNN(nn.Module):
    # Configurable RSNN for all variants B0-B7.
    # neuron_type: 'lif' or 'alif'
    # heterogeneous_tau: True for B2
    # n_hidden_layers: 2 for B3
    # input_bn: True for B4
    # readout_type: 'attention' for B5
    # hidden_size: 512 for B6
    def __init__(self, input_size, hidden_size=256, output_size=5,
                 n_hidden_layers=1, recurrent=True,
                 tau_mem=20.0, tau_syn=10.0, dt=10.0,
                 learnable_tau=True, dropout=0.3,
                 neuron_type='lif', heterogeneous_tau=False,
                 input_bn=False, readout_type='max_over_time',
                 tau_adapt=100.0, beta_adapt=0.1):
        super().__init__()
        self.readout_type = readout_type
        self.hidden_size = hidden_size
        self.input_bn_flag = input_bn
        if input_bn:
            self.bn = nn.BatchNorm1d(input_size)

        LayerClass = ALIFLayer if neuron_type == 'alif' else LIFLayer
        layers = []
        for i in range(n_hidden_layers):
            in_sz = input_size if i == 0 else hidden_size
            kw = dict(input_size=in_sz, hidden_size=hidden_size,
                      recurrent=recurrent, tau_mem_init=tau_mem,
                      tau_syn_init=tau_syn, dt=dt, learnable_tau=learnable_tau,
                      dropout=dropout, heterogeneous_tau=heterogeneous_tau)
            if neuron_type == 'alif':
                kw['tau_adapt'] = tau_adapt
                kw['beta_adapt'] = beta_adapt
            layers.append(LayerClass(**kw))
        self.hidden_layers = nn.ModuleList(layers)

        if readout_type == 'attention':
            self.readout = AttentionReadout(hidden_size, output_size)
        else:
            self.readout = ReadoutLayer(hidden_size, output_size, tau_mem=tau_mem, dt=dt)

    def forward(self, x):
        if self.input_bn_flag:
            B, T, C = x.shape
            x = self.bn(x.reshape(B * T, C)).reshape(B, T, C)
        all_spikes = []
        h = x
        for layer in self.hidden_layers:
            spikes, _ = layer(h)
            all_spikes.append(spikes)
            h = spikes
        if self.readout_type == 'attention':
            logits_seq, attn_w = self.readout(h)
            weighted = (logits_seq * attn_w).sum(dim=1)
            return weighted, all_spikes, logits_seq
        else:
            out_mem = self.readout(h)
            if self.readout_type == 'max_over_time':
                output, _ = torch.max(out_mem, dim=1)
            elif self.readout_type == 'last_timestep':
                output = out_mem[:, -1, :]
            else:
                raise ValueError(f'Unknown readout: {self.readout_type}')
            return output, all_spikes, out_mem

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

### 2.6 Baselines: LSTM and 1D-CNN

In [9]:
class LSTMBaseline(nn.Module):
    def __init__(self, input_size, hidden_size=128, n_layers=2,
                 output_size=5, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])
    def forward_seq(self, x):
        out, _ = self.lstm(x)
        return out
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class CNNBaseline(nn.Module):
    def __init__(self, input_channels, output_size=5):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, 64, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(128, output_size)
    def forward(self, x):
        h = self.conv(x.transpose(1, 2))
        return self.fc(h.squeeze(-1))
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## 3. Training Engine

Spike regularisation (Cramer et al. 2020 Eqs 10-11), gradient clipping,
early stopping on validation, checkpoint saving.

In [10]:
def spike_regularization(all_spikes, theta_l=0.01, s_l=1.0, theta_u=100.0, s_u=0.06):
    reg = torch.tensor(0.0, device=all_spikes[0].device)
    for spk in all_spikes:
        B, T, N = spk.shape
        mean_rate = spk.sum(dim=1) / T
        reg += s_l / (B * N) * (F.relu(theta_l - mean_rate) ** 2).sum()
        pop_count = spk.sum(dim=(1, 2)) / N
        reg += s_u / B * (F.relu(pop_count - theta_u) ** 2).sum()
    return reg


def train_snn(model, train_ld, val_ld, test_ld, n_epochs=80, lr=1e-3,
              device='cuda', patience=20, verbose_every=10):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val, best_state, wait = 0, None, 0
    history = {'train_loss':[], 'train_acc':[], 'val_acc':[], 'test_acc':[]}
    t0 = time.time()
    for epoch in range(n_epochs):
        model.train()
        tot_loss, correct, total = 0, 0, 0
        for x, y in train_ld:
            x, y = x.to(device), y.to(device)
            logits, spks, _ = model(x)
            loss = criterion(logits, y)
            if spks:
                loss += spike_regularization(spks)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tot_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            total += len(y)
        train_acc = correct / total
        val_acc = eval_snn(model, val_ld, device)
        test_acc = eval_snn(model, test_ld, device)
        history['train_loss'].append(tot_loss/total)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['test_acc'].append(test_acc)
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if epoch % verbose_every == 0 or epoch == n_epochs - 1:
            m = '*' if wait == 0 else ''
            print(f'  Ep {epoch:3d}: loss={tot_loss/total:.4f} tr={train_acc:.4f} va={val_acc:.4f} te={test_acc:.4f} {m}')
        if wait >= patience:
            print(f'  Early stop at epoch {epoch}')
            break
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    final = eval_snn(model, test_ld, device)
    elapsed = time.time() - t0
    print(f'  Final test: {final*100:.2f}% ({elapsed:.0f}s)')
    return final, history, model, elapsed


@torch.no_grad()
def eval_snn(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits, _, _ = model(x)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return correct / total


def train_baseline(model, train_ld, val_ld, test_ld, n_epochs=80, lr=1e-3,
                   device='cuda', patience=20, verbose_every=20):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val, best_state, wait = 0, None, 0
    t0 = time.time()
    for epoch in range(n_epochs):
        model.train()
        for x, y in train_ld:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        val_acc = eval_bl(model, val_ld, device)
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if epoch % verbose_every == 0 or epoch == n_epochs - 1:
            te = eval_bl(model, test_ld, device)
            print(f'  Ep {epoch}: val={val_acc:.4f} test={te:.4f}')
        if wait >= patience:
            print(f'  Early stop at epoch {epoch}'); break
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    final = eval_bl(model, test_ld, device)
    print(f'  Final: {final*100:.2f}% ({time.time()-t0:.0f}s)')
    return final, model, time.time()-t0

@torch.no_grad()
def eval_bl(model, loader, device):
    model.eval()
    c, t = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        c += (model(x).argmax(1) == y).sum().item(); t += len(y)
    return c / t

In [11]:
def save_checkpoint(model, config, acc, history, elapsed, arch_name, ds_name):
    path = os.path.join(SAVE_DIR, f'{arch_name}_{ds_name}.pth')
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config,
        'test_accuracy': acc,
        'history': history,
        'training_time_s': elapsed,
        'arch_name': arch_name,
        'dataset_name': ds_name,
    }, path)
    print(f'  Saved: {path}')
    return path

def run_arch(name, config, tr, va, te, in_sz, out_sz, ds, epochs=80, lr=1e-3):
    print(f'\n{"="*65}')
    print(f'  {name} on {ds}')
    print(f'{"="*65}')
    model = SNN(input_size=in_sz, output_size=out_sz, **config)
    print(f'  Params: {model.count_params():,}')
    acc, hist, model, elapsed = train_snn(model, tr, va, te, epochs, lr, device, patience=20)
    full_cfg = {**config, 'input_size': in_sz, 'output_size': out_sz}
    save_checkpoint(model, full_cfg, acc, hist, elapsed, name, ds)
    ALL_RESULTS[f'{name}_{ds}'] = {'accuracy': acc, 'params': model.count_params(),
                                    'time_s': elapsed, 'config': full_cfg}
    return acc, model

## 4. Architecture Configurations

Each row = exactly one change from base.

In [12]:
BASE = dict(
    hidden_size=256, n_hidden_layers=1, recurrent=True,
    tau_mem=20.0, tau_syn=10.0, dt=10.0,
    learnable_tau=True, dropout=0.3,
    neuron_type='lif', heterogeneous_tau=False,
    input_bn=False, readout_type='max_over_time',
)

ARCHS = OrderedDict([
    ('B0_base',      {**BASE}),
    ('B1_alif',      {**BASE, 'neuron_type': 'alif', 'tau_adapt': 100.0, 'beta_adapt': 0.1}),
    ('B2_het_tau',   {**BASE, 'heterogeneous_tau': True}),
    ('B3_2layer',    {**BASE, 'n_hidden_layers': 2}),
    ('B4_input_bn',  {**BASE, 'input_bn': True}),
    ('B5_attention', {**BASE, 'readout_type': 'attention'}),
    ('B6_scale512',  {**BASE, 'hidden_size': 512}),
])

print('Architectures:')
descs = {'B0_base':'Control', 'B1_alif':'LIF->ALIF', 'B2_het_tau':'Per-neuron tau',
         'B3_2layer':'2 hidden layers', 'B4_input_bn':'Input BatchNorm',
         'B5_attention':'Attention readout', 'B6_scale512':'hidden 256->512'}
for k,v in descs.items():
    print(f'  {k:18s} {v}')

Architectures:
  B0_base            Control
  B1_alif            LIF->ALIF
  B2_het_tau         Per-neuron tau
  B3_2layer          2 hidden layers
  B4_input_bn        Input BatchNorm
  B5_attention       Attention readout
  B6_scale512        hidden 256->512


## 5. Dataset: SHD

Event-based audio. 700 channels, 100 time bins, 20 classes.
Add to Kaggle: 'shd-snns-dataset' by harshitad879.

In [13]:
import h5py

SHD_DIR = '/kaggle/input/datasets/harshitad879/shd-snns-dataset'

class SHDDataset(Dataset):
    def __init__(self, h5_path, n_time_bins=100, n_channels=700, max_time=1.0):
        self.n_time_bins = n_time_bins
        self.n_channels = n_channels
        self.max_time = max_time
        with h5py.File(h5_path, 'r') as f:
            self.times = [f['spikes']['times'][i] for i in range(len(f['spikes']['times']))]
            self.units = [f['spikes']['units'][i] for i in range(len(f['spikes']['units']))]
            self.labels = f['labels'][:]
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        times = self.times[idx]
        units = self.units[idx].astype(np.int64)
        frame = np.zeros((self.n_time_bins, self.n_channels), dtype=np.float32)
        valid = (times >= 0) & (times < self.max_time) & (units >= 0) & (units < self.n_channels)
        t_idx = np.minimum((times[valid] / self.max_time * self.n_time_bins).astype(int), self.n_time_bins - 1)
        np.add.at(frame, (t_idx, units[valid]), 1.0)
        return torch.tensor(frame), int(self.labels[idx])

train_ds_shd = SHDDataset(os.path.join(SHD_DIR, 'shd_train.h5'))
test_ds_shd = SHDDataset(os.path.join(SHD_DIR, 'shd_test.h5'))
n_val = int(0.1 * len(train_ds_shd))
tr_sub, va_sub = torch.utils.data.random_split(train_ds_shd, [len(train_ds_shd)-n_val, n_val],
    generator=torch.Generator().manual_seed(SEED))
shd_tr = DataLoader(tr_sub, batch_size=256, shuffle=True, num_workers=2)
shd_va = DataLoader(va_sub, batch_size=256, shuffle=False, num_workers=2)
shd_te = DataLoader(test_ds_shd, batch_size=256, shuffle=False, num_workers=2)
SHD_IN, SHD_CLS = 700, 20
print(f'SHD loaded: input={SHD_IN}, classes={SHD_CLS}')

SHD loaded: input=700, classes=20


## 6. Dataset: PTB-XL (ECG)

12-lead ECG, 10s at 100Hz, 5 classes. Delta modulation encoding.
Add to Kaggle: 'ptb-xl-dataset' by khyeh0719.

In [14]:
import wfdb, ast, pandas as pd, glob

# Auto-find ptbxl_database.csv wherever it lives
ecg_csv = glob.glob('/kaggle/input/**/ptbxl_database.csv', recursive=True)
assert ecg_csv, "ptbxl_database.csv not found - check dataset is attached"
ECG_DIR = os.path.dirname(ecg_csv[0])
print(f"ECG_DIR: {ECG_DIR}")

df_ecg = pd.read_csv(os.path.join(ECG_DIR, 'ptbxl_database.csv'))
df_ecg.scp_codes = df_ecg.scp_codes.apply(ast.literal_eval)
scp_df = pd.read_csv(os.path.join(ECG_DIR, 'scp_statements.csv'), index_col=0)
scp_df = scp_df[scp_df.diagnostic == 1.0]
SUPERCLASSES = ['NORM','MI','STTC','CD','HYP']
c2i = {c:i for i,c in enumerate(SUPERCLASSES)}
def get_sc(scp):
    best, lk = None, 0
    for k,v in scp.items():
        if k in scp_df.index:
            sc = scp_df.loc[k].diagnostic_class
            if sc in SUPERCLASSES and v > lk: best, lk = sc, v
    return best
df_ecg['sc'] = df_ecg.scp_codes.apply(get_sc)
df_ecg = df_ecg.dropna(subset=['sc'])
df_ecg['label'] = df_ecg.sc.map(c2i)
def load_ecg(df_sub, sr=100):
    sigs, labs = [], []
    for _, r in df_sub.iterrows():
        try:
            p = os.path.join(ECG_DIR, r['filename_lr'])
            if p.endswith('.hea'): p = p[:-4]
            sigs.append(wfdb.rdrecord(p).p_signal.astype(np.float32))
            labs.append(r['label'])
        except: pass
    return np.array(sigs), np.array(labs)
print('Loading ECG...')
Xtr_e, ytr_e = load_ecg(df_ecg[df_ecg.strat_fold.isin(range(1,9))])
Xva_e, yva_e = load_ecg(df_ecg[df_ecg.strat_fold==9])
Xte_e, yte_e = load_ecg(df_ecg[df_ecg.strat_fold==10])
mu_e = Xtr_e.mean(axis=(0,1)); sd_e = Xtr_e.std(axis=(0,1))+1e-8
Xtr_e = (Xtr_e-mu_e)/sd_e; Xva_e = (Xva_e-mu_e)/sd_e; Xte_e = (Xte_e-mu_e)/sd_e
print(f'ECG: tr={Xtr_e.shape}, va={Xva_e.shape}, te={Xte_e.shape}')

ECG_DIR: /kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1


Loading ECG...


ECG: tr=(17100, 1000, 12), va=(2155, 1000, 12), te=(2162, 1000, 12)


In [15]:
class ECGDataset(Dataset):
    def __init__(self, X, y, enc='delta', n_bins=250, **kw):
        self.X, self.y = X, torch.tensor(y, dtype=torch.long)
        self.enc, self.n_bins, self.kw = enc, n_bins, kw
        self.input_size = self._enc(X[0]).shape[1]
        self.n_classes = len(np.unique(y))
    def _enc(self, sig):
        idx = np.linspace(0, sig.shape[0]-1, self.n_bins).astype(int)
        s = sig[idx]
        if self.enc=='direct': return s
        elif self.enc=='delta': return SpikeEncoder.delta_modulation(s, **self.kw)
        elif self.enc=='adaptive_delta':
            e, _ = SpikeEncoder.adaptive_delta(s, **self.kw); return e
        return s
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return torch.tensor(self._enc(self.X[i]), dtype=torch.float32), self.y[i]

# Delta mod loaders (SNN)
ecg_tr = DataLoader(ECGDataset(Xtr_e,ytr_e,'delta',250,threshold=0.1), 128, True, num_workers=2)
ecg_va = DataLoader(ECGDataset(Xva_e,yva_e,'delta',250,threshold=0.1), 128, False, num_workers=2)
ecg_te = DataLoader(ECGDataset(Xte_e,yte_e,'delta',250,threshold=0.1), 128, False, num_workers=2)
# Direct loaders (baselines)
ecg_tr_d = DataLoader(ECGDataset(Xtr_e,ytr_e,'direct',250), 128, True, num_workers=2)
ecg_va_d = DataLoader(ECGDataset(Xva_e,yva_e,'direct',250), 128, False, num_workers=2)
ecg_te_d = DataLoader(ECGDataset(Xte_e,yte_e,'direct',250), 128, False, num_workers=2)
ECG_IN = ECGDataset(Xtr_e[:1],ytr_e[:1],'delta',250,threshold=0.1).input_size
ECG_CLS = 5
print(f'ECG loaders ready. Input={ECG_IN}, Classes={ECG_CLS}')

ECG loaders ready. Input=12, Classes=5


## 7. Dataset: FI-2010 (LOB)

40 features, 100-event sequences, 3 classes. Two loaders: direct + adaptive delta.
Add to Kaggle: 'fi-2010' by ulfricirons.

In [16]:
import glob

# Auto-find FI-2010 training file wherever it lives
fi_train_file = glob.glob('/kaggle/input/**/Train_Dst_NoAuction_DecPre_CF_7.txt', recursive=True)
assert fi_train_file, "FI-2010 training file not found - check dataset is attached"
FI_DIR = os.path.dirname(fi_train_file[0])
print(f"FI_DIR: {FI_DIR}")

SEQ_LEN = 100
def prep_x(d): return d[:40,:].T.astype(np.float32)
def get_lab(d): return (d[-5:,:].T.astype(int) - 1)
def make_seq(X, y, sl):
    n = len(X) - sl + 1
    return np.array([X[i:i+sl] for i in range(n)]).astype(np.float32), y[sl-1:]

train_raw = np.loadtxt(os.path.join(FI_DIR, 'Train_Dst_NoAuction_DecPre_CF_7.txt'))

# Auto-find test files (day 8 and 9)
test8_file = glob.glob(os.path.join(FI_DIR, '**', 'Test_Dst_NoAuction_DecPre_CF_8.txt'), recursive=True)
test9_file = glob.glob(os.path.join(FI_DIR, '**', 'Test_Dst_NoAuction_DecPre_CF_9.txt'), recursive=True)

# If test files are in the same dir, use directly; otherwise search broader
if not test8_file:
    test8_file = glob.glob('/kaggle/input/**/Test_Dst_NoAuction_DecPre_CF_8.txt', recursive=True)
if not test9_file:
    test9_file = glob.glob('/kaggle/input/**/Test_Dst_NoAuction_DecPre_CF_9.txt', recursive=True)

assert test8_file, "Test file day 8 not found"
assert test9_file, "Test file day 9 not found"

test8 = np.loadtxt(test8_file[0])
test9 = np.loadtxt(test9_file[0])

Xtr_f, ytr_f = make_seq(prep_x(train_raw), get_lab(train_raw)[:,0], SEQ_LEN)
Xva_f, yva_f = make_seq(prep_x(test8), get_lab(test8)[:,0], SEQ_LEN)
Xte_f, yte_f = make_seq(prep_x(test9), get_lab(test9)[:,0], SEQ_LEN)
mu_f = Xtr_f.mean(axis=(0,1)); sd_f = Xtr_f.std(axis=(0,1))+1e-8
Xtr_f=(Xtr_f-mu_f)/sd_f; Xva_f=(Xva_f-mu_f)/sd_f; Xte_f=(Xte_f-mu_f)/sd_f
print(f'FI-2010: tr={Xtr_f.shape}, va={Xva_f.shape}, te={Xte_f.shape}')
print(f'Labels (test): {np.bincount(yte_f)}')

FI_DIR: /kaggle/input/datasets/ulfricirons/fi-2010/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Training


FI-2010: tr=(254651, 100, 40), va=(52073, 100, 40), te=(31838, 100, 40)
Labels (test): [ 5510 21311  5017]


In [17]:
class LOBDataset(Dataset):
    def __init__(self, X, y, enc='direct', **kw):
        self.X, self.y = X, torch.tensor(y, dtype=torch.long)
        self.enc, self.kw = enc, kw
        self.input_size = self._enc(X[0]).shape[1]
    def _enc(self, s):
        if self.enc=='direct': return s
        elif self.enc=='delta': return SpikeEncoder.delta_modulation(s, **self.kw)
        elif self.enc=='adaptive_delta':
            e, _ = SpikeEncoder.adaptive_delta(s, **self.kw); return e
        return s
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return torch.tensor(self._enc(self.X[i]), dtype=torch.float32), self.y[i]

# Direct
fi_tr = DataLoader(LOBDataset(Xtr_f,ytr_f,'direct'), 256, True, num_workers=2)
fi_va = DataLoader(LOBDataset(Xva_f,yva_f,'direct'), 256, False, num_workers=2)
fi_te = DataLoader(LOBDataset(Xte_f,yte_f,'direct'), 256, False, num_workers=2)
# Adaptive delta
fi_tr_ad = DataLoader(LOBDataset(Xtr_f,ytr_f,'adaptive_delta',percentile=90), 256, True, num_workers=2)
fi_va_ad = DataLoader(LOBDataset(Xva_f,yva_f,'adaptive_delta',percentile=90), 256, False, num_workers=2)
fi_te_ad = DataLoader(LOBDataset(Xte_f,yte_f,'adaptive_delta',percentile=90), 256, False, num_workers=2)
FI_IN, FI_CLS = 40, 3
sd_fi = LOBDataset(Xtr_f[:1],ytr_f[:1],'adaptive_delta',percentile=90)
print(f'FI-2010 adaptive delta spike density: {(torch.tensor(sd_fi._enc(Xtr_f[0])).abs()>0).float().mean()*100:.1f}%')
print(f'FI-2010 loaders ready. Input={FI_IN}, Classes={FI_CLS}')

FI-2010 adaptive delta spike density: 10.6%
FI-2010 loaders ready. Input=40, Classes=3


## 8. Baselines (LSTM + CNN, all 3 datasets)

In [18]:
print('='*65)
print('BASELINES')
print('='*65)
br = {}

print('\n--- SHD LSTM ---')
acc, m, _ = train_baseline(LSTMBaseline(700,128,2,20,0.2), shd_tr, shd_va, shd_te)
br['lstm_shd'] = acc; torch.save(m.state_dict(), os.path.join(SAVE_DIR,'lstm_shd.pth'))

print('\n--- SHD CNN ---')
acc, m, _ = train_baseline(CNNBaseline(700,20), shd_tr, shd_va, shd_te)
br['cnn_shd'] = acc; torch.save(m.state_dict(), os.path.join(SAVE_DIR,'cnn_shd.pth'))

print('\n--- ECG LSTM ---')
acc, m, _ = train_baseline(LSTMBaseline(12,128,2,5,0.3), ecg_tr_d, ecg_va_d, ecg_te_d)
br['lstm_ecg'] = acc; torch.save(m.state_dict(), os.path.join(SAVE_DIR,'lstm_ecg.pth'))

print('\n--- ECG CNN ---')
acc, m, _ = train_baseline(CNNBaseline(12,5), ecg_tr_d, ecg_va_d, ecg_te_d)
br['cnn_ecg'] = acc; torch.save(m.state_dict(), os.path.join(SAVE_DIR,'cnn_ecg.pth'))

print('\n--- FI-2010 LSTM ---')
acc, m, _ = train_baseline(LSTMBaseline(40,128,2,3,0.3), fi_tr, fi_va, fi_te)
br['lstm_fi'] = acc; torch.save(m.state_dict(), os.path.join(SAVE_DIR,'lstm_fi.pth'))

print('\n--- FI-2010 CNN ---')
acc, m, _ = train_baseline(CNNBaseline(40,3), fi_tr, fi_va, fi_te)
br['cnn_fi'] = acc; torch.save(m.state_dict(), os.path.join(SAVE_DIR,'cnn_fi.pth'))

print('\nBaselines:')
for k,v in br.items(): print(f'  {k}: {v*100:.2f}%')
ALL_RESULTS['baselines'] = br

BASELINES

--- SHD LSTM ---


  Ep 0: val=0.0528 test=0.0521


  Ep 20: val=0.7252 test=0.6250


  Ep 40: val=0.8503 test=0.6652


  Ep 60: val=0.8822 test=0.6873


...

  Ep 0: val=0.7036 test=0.6694


  Ep 20: val=0.7968 test=0.7723


  Ep 40: val=0.7889 test=0.7686


  Early stop at epoch 42


  Final: 77.06% (397s)

Baselines:
  lstm_shd: 71.69%
  cnn_shd: 84.94%
  lstm_ecg: 68.41%
  cnn_ecg: 70.03%
  lstm_fi: 78.06%
  cnn_fi: 77.06%


## 9. Architecture Sweep: SHD

In [19]:
print('='*65); print('SWEEP: SHD'); print('='*65)
shd_res, shd_mod = {}, {}
for name, cfg in ARCHS.items():
    a, m = run_arch(name, cfg, shd_tr, shd_va, shd_te, SHD_IN, SHD_CLS, 'shd', 80)
    shd_res[name] = a; shd_mod[name] = m
print('\nSHD results:')
for n,a in sorted(shd_res.items(), key=lambda x:x[1], reverse=True):
    print(f'  {n}: {a*100:.2f}%')

SWEEP: SHD

  B0_base on shd
  Params: 250,368


  Ep   0: loss=2.8632 tr=0.1436 va=0.2025 te=0.1780 *


  Ep  10: loss=1.3150 tr=0.5610 va=0.5080 te=0.5269 


  Ep  20: loss=0.9416 tr=0.6827 va=0.6638 te=0.5919 


  Ep  30: loss=0.7422 tr=0.7483 va=0.6945 te=0.6307 


...

  Ep  50: loss=0.5069 tr=0.8323 va=0.8074 te=0.6581 *


  Ep  60: loss=0.5218 tr=0.8266 va=0.7988 te=0.6568 


  Ep  70: loss=0.4868 tr=0.8423 va=0.7963 te=0.6630 


  Ep  79: loss=0.4657 tr=0.8553 va=0.7926 te=0.6599 


  Final test: 66.74% (1007s)
  Saved: /kaggle/working/checkpoints/B6_scale512_shd.pth

SHD results:
  B5_attention: 77.25%
  B3_2layer: 75.75%
  B2_het_tau: 75.49%
  B6_scale512: 66.74%
  B0_base: 66.39%
  B1_alif: 54.02%
  B4_input_bn: 51.63%


## 10. Architecture Sweep: ECG

In [20]:
print('='*65); print('SWEEP: ECG'); print('='*65)
ecg_res, ecg_mod = {}, {}
for name, cfg in ARCHS.items():
    a, m = run_arch(name, cfg, ecg_tr, ecg_va, ecg_te, ECG_IN, ECG_CLS, 'ecg', 100)
    ecg_res[name] = a; ecg_mod[name] = m
print('\nECG results:')
for n,a in sorted(ecg_res.items(), key=lambda x:x[1], reverse=True):
    print(f'  {n}: {a*100:.2f}%')

SWEEP: ECG

  B0_base on ecg
  Params: 70,400


  Ep   0: loss=1.3115 tr=0.4856 va=0.5364 te=0.5296 *


  Ep  10: loss=1.0088 tr=0.6212 va=0.6000 te=0.6055 


  Ep  20: loss=0.9779 tr=0.6378 va=0.6102 te=0.6105 


  Ep  30: loss=0.9755 tr=0.6385 va=0.6213 te=0.6244 


...

  Ep   0: loss=1.2182 tr=0.5388 va=0.5726 te=0.5527 *


  Ep  10: loss=1.0676 tr=0.5975 va=0.5661 te=0.5685 


  Ep  20: loss=1.0524 tr=0.6087 va=0.5763 te=0.5805 


  Early stop at epoch 27


  Final test: 59.30% (2417s)
  Saved: /kaggle/working/checkpoints/B6_scale512_ecg.pth

ECG results:
  B5_attention: 65.73%
  B4_input_bn: 63.09%
  B2_het_tau: 62.44%
  B3_2layer: 62.40%
  B1_alif: 62.26%
  B0_base: 61.93%
  B6_scale512: 59.30%


## 11. Architecture Sweep: FI-2010

B1 (ALIF) also runs on adaptive delta input — on direct input, neurons barely fire
so raising the threshold via ALIF makes it worse.

In [21]:
print('='*65); print('SWEEP: FI-2010'); print('='*65)
fi_res, fi_mod = {}, {}
for name, cfg in ARCHS.items():
    a, m = run_arch(name, cfg, fi_tr, fi_va, fi_te, FI_IN, FI_CLS, 'fi2010', 80)
    fi_res[name] = a; fi_mod[name] = m
    if name == 'B1_alif':
        a2, m2 = run_arch('B1_alif_adap', cfg, fi_tr_ad, fi_va_ad, fi_te_ad,
                          FI_IN, FI_CLS, 'fi2010', 80)
        fi_res['B1_alif_adap'] = a2; fi_mod['B1_alif_adap'] = m2
print('\nFI-2010 results:')
for n,a in sorted(fi_res.items(), key=lambda x:x[1], reverse=True):
    print(f'  {n}: {a*100:.2f}%')

SWEEP: FI-2010

  B0_base on fi2010
  Params: 77,056


  Ep   0: loss=0.9301 tr=0.6051 va=0.7036 te=0.6694 *


## 12. B7: Best Combination

Top-2 modifications per dataset, combined. Data-driven (exploratory).

In [ ]:
def top2(res, base):
    gains = {k: v-base for k,v in res.items() if k != 'B0_base' and not k.endswith('_adap')}
    return sorted(gains.items(), key=lambda x:x[1], reverse=True)[:2]

def merge(n1, n2):
    m = dict(BASE)
    for k,v in ARCHS.get(n1, {}).items():
        if v != BASE.get(k): m[k] = v
    for k,v in ARCHS.get(n2, {}).items():
        if v != BASE.get(k) and m.get(k) == BASE.get(k): m[k] = v
    return m

print('='*65); print('B7: BEST COMBO'); print('='*65)

t2s = top2(shd_res, shd_res['B0_base'])
print(f'SHD top-2: {t2s[0][0]} (+{t2s[0][1]*100:.2f}pp), {t2s[1][0]} (+{t2s[1][1]*100:.2f}pp)')
a, m = run_arch('B7_combo', merge(t2s[0][0],t2s[1][0]), shd_tr, shd_va, shd_te, SHD_IN, SHD_CLS, 'shd', 80)
shd_res['B7_combo'] = a

t2e = top2(ecg_res, ecg_res['B0_base'])
print(f'ECG top-2: {t2e[0][0]} (+{t2e[0][1]*100:.2f}pp), {t2e[1][0]} (+{t2e[1][1]*100:.2f}pp)')
a, m = run_arch('B7_combo', merge(t2e[0][0],t2e[1][0]), ecg_tr, ecg_va, ecg_te, ECG_IN, ECG_CLS, 'ecg', 100)
ecg_res['B7_combo'] = a

t2f = top2(fi_res, fi_res['B0_base'])
print(f'FI top-2: {t2f[0][0]} (+{t2f[0][1]*100:.2f}pp), {t2f[1][0]} (+{t2f[1][1]*100:.2f}pp)')
a, m = run_arch('B7_combo', merge(t2f[0][0],t2f[1][0]), fi_tr, fi_va, fi_te, FI_IN, FI_CLS, 'fi2010', 80)
fi_res['B7_combo'] = a

## 13. Final Summary + Save

In [ ]:
print('='*75)
print('COMPLETE ARCHITECTURE LAB RESULTS')
print('='*75)

for ds, res, la, ca in [('SHD',shd_res,br['lstm_shd'],br['cnn_shd']),
                         ('ECG',ecg_res,br['lstm_ecg'],br['cnn_ecg']),
                         ('FI-2010',fi_res,br['lstm_fi'],br['cnn_fi'])]:
    print(f'\n--- {ds} ---')
    base = res.get('B0_base', 0)
    for n,a in sorted(res.items(), key=lambda x:x[1], reverse=True):
        best = ' *' if a == max(res.values()) else ''
        print(f'  {n:20s} {a*100:7.2f}%  vs base: {(a-base)*100:+6.2f}pp  vs LSTM: {(a-la)*100:+6.2f}pp{best}')
    print(f'  {"LSTM":20s} {la*100:7.2f}%')
    print(f'  {"CNN":20s} {ca*100:7.2f}%')

best_per = {}
for ds, res in [('shd',shd_res),('ecg',ecg_res),('fi2010',fi_res)]:
    b = max(res, key=res.get)
    best_per[ds] = {'arch': b, 'accuracy': res[b]}
    print(f'\nBest for {ds}: {b} ({res[b]*100:.2f}%)')

In [ ]:
master = {
    'baselines': br,
    'shd': shd_res, 'ecg': ecg_res, 'fi2010': fi_res,
    'best_per_dataset': best_per,
    'architecture_configs': {k:str(v) for k,v in ARCHS.items()},
}
with open(os.path.join(SAVE_DIR, 'architecture_results.json'), 'w') as f:
    json.dump(master, f, indent=2, default=str)
print(f'Saved architecture_results.json')
print('\nCheckpoints:')
for fn in sorted(os.listdir(SAVE_DIR)):
    sz = os.path.getsize(os.path.join(SAVE_DIR,fn))/1e6
    print(f'  {fn}: {sz:.1f}MB')

In [ ]:
# Comparison plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (ds, res, la, _) in zip(axes, [('SHD',shd_res,br['lstm_shd'],0),
                                        ('ECG',ecg_res,br['lstm_ecg'],0),
                                        ('FI-2010',fi_res,br['lstm_fi'],0)]):
    names = list(res.keys())
    accs = [res[n]*100 for n in names]
    cols = ['steelblue' if n!='B0_base' else 'gray' for n in names]
    ax.barh(names, accs, color=cols, alpha=0.8)
    ax.axvline(x=la*100, color='coral', linestyle='--', label=f'LSTM {la*100:.1f}%')
    ax.set_xlabel('Test Accuracy (%)'); ax.set_title(ds)
    ax.legend(fontsize=8); ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'architecture_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved architecture_comparison.png')